In [ ]:
import sys
IN_COLAB = ('google.colab' in sys.modules)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'tensorflow', 'transformers', 'torch', 'torchvision',
                    'seaborn', 'scikit-learn'])
print('Environment ready.')


# Aircraft Damage Classification & Captioning

**Binary classification of aircraft structural damage (crack vs. dent) using EfficientNetB0 transfer learning, with BLIP-based natural-language captioning.**

---

## 1. Environment Setup

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import pathlib
for d in ['../outputs/figures', '../outputs/models']:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print('Output dirs ready.')

## 2. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc
)

tf.random.set_seed(42)
np.random.seed(42)
print(f'TensorFlow {tf.__version__} | NumPy {np.__version__}')

## 3. Configuration

In [ ]:
DATA_DIR   = Path('../aircraft_damage_dataset_v1')
TRAIN_DIR  = DATA_DIR / 'train'
VALID_DIR  = DATA_DIR / 'valid'
TEST_DIR   = DATA_DIR / 'test'
FIG_DIR    = Path('../outputs/figures')
MODEL_DIR  = Path('../outputs/models')

CLASSES    = ['crack', 'dent']
IMG_SIZE   = (224, 224)
BATCH_SIZE = 16
EPOCHS_P1  = 15
EPOCHS_P2  = 30
UNFREEZE_N = 20

print('Config loaded.')
for name, path in [('Train', TRAIN_DIR), ('Valid', VALID_DIR), ('Test', TEST_DIR)]:
    print(f'  {name}: {path}')

## 4. Exploratory Data Analysis

In [ ]:
def count_images(root):
    return {cls: len(list((root / cls).glob('*.jpg'))) for cls in CLASSES}

splits = {'train': count_images(TRAIN_DIR),
          'valid': count_images(VALID_DIR),
          'test' : count_images(TEST_DIR)}

print(f"  {'Split':<8} {'crack':>8} {'dent':>8} {'total':>8}")
print('  ' + '-'*30)
for split, counts in splits.items():
    total = sum(counts.values())
    print(f"  {split:<8} {counts['crack']:>8} {counts['dent']:>8} {total:>8}")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(splits))
w = 0.3
for i, cls in enumerate(CLASSES):
    ax.bar(x + i*w, [splits[s][cls] for s in splits], w, label=cls)
ax.set_xticks(x + w/2)
ax.set_xticklabels(splits.keys())
ax.set_ylabel('Image count')
ax.set_title('Class distribution per split')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'class_distribution.png', dpi=150)
plt.show()

## 5. Sample Images

In [ ]:
from PIL import Image

def show_samples(root, classes, n_per_class=4):
    fig, axes = plt.subplots(len(classes), n_per_class,
                             figsize=(n_per_class * 3, len(classes) * 3))
    for r, cls in enumerate(classes):
        imgs = sorted((root / cls).glob('*.jpg'))[:n_per_class]
        for c, img_path in enumerate(imgs):
            ax = axes[r][c]
            ax.imshow(Image.open(img_path))
            ax.set_title(cls if c == 0 else '', fontsize=11, fontweight='bold')
            ax.axis('off')
    plt.suptitle('Sample training images', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'sample_images.png', dpi=150)
    plt.show()

show_samples(TRAIN_DIR, CLASSES)

## 6. Data Preprocessing & Augmentation

Augmentation is applied **only to training data**. Val/test use plain rescaling so metrics reflect real-world performance.
Augmentation simulates variability in real inspection photos (lighting, angle, zoom).

In [ ]:
def build_generators(train_dir, valid_dir, test_dir, img_size, batch_size):
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.15,
        horizontal_flip=True,
        brightness_range=[0.8, 1.2],
        fill_mode='nearest',
    )
    eval_datagen = ImageDataGenerator(rescale=1.0 / 255)

    train_gen = train_datagen.flow_from_directory(
        train_dir, target_size=img_size, batch_size=batch_size,
        class_mode='binary', seed=42,
    )
    valid_gen = eval_datagen.flow_from_directory(
        valid_dir, target_size=img_size, batch_size=batch_size,
        class_mode='binary', seed=42,
    )
    test_gen = eval_datagen.flow_from_directory(
        test_dir, target_size=img_size, batch_size=batch_size,
        class_mode='binary', shuffle=False,
    )
    return train_gen, valid_gen, test_gen

train_gen, valid_gen, test_gen = build_generators(
    TRAIN_DIR, VALID_DIR, TEST_DIR, IMG_SIZE, BATCH_SIZE
)
print(f'Train batches : {len(train_gen)}')
print(f'Valid batches : {len(valid_gen)}')
print(f'Test  batches : {len(test_gen)}')
print(f'Class indices : {train_gen.class_indices}')

## 7. Model Architecture: EfficientNetB0

We use **EfficientNetB0** (5.3M params) instead of VGG16 (138M params).
It achieves comparable accuracy with 26x fewer parameters, critical for small datasets where large models overfit easily.

**Two-phase training strategy:**
1. Freeze the base, train only the classification head
2. Unfreeze the top N layers and fine-tune at a very low learning rate

In [ ]:
def build_model(img_size=(224, 224)):
    base = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(*img_size, 3),
    )
    base.trainable = False  # freeze for phase 1

    inputs = tf.keras.Input(shape=(*img_size, 3))
    # training=False keeps BatchNorm in inference mode during phase 1
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='aircraft_damage_classifier')
    return model, base

model, base_model = build_model(IMG_SIZE)
model.summary()

## 8. Training Callbacks

In [ ]:
def get_callbacks(checkpoint_name):
    return [
        EarlyStopping(
            monitor='val_loss', patience=7,
            restore_best_weights=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.3,
            patience=3, min_lr=1e-7, verbose=1
        ),
        ModelCheckpoint(
            str(MODEL_DIR / checkpoint_name),
            monitor='val_loss', save_best_only=True, verbose=1
        ),
    ]

## 9. Phase 1: Train Classification Head (Base Frozen)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

print('Phase 1: training classification head (base frozen)...')
history_p1 = model.fit(
    train_gen,
    validation_data=valid_gen,
    epochs=EPOCHS_P1,
    callbacks=get_callbacks('best_phase1.keras'),
    verbose=1,
)

## 10. Phase 2: Fine-Tune Top Layers

Unfreeze the top 20 layers of EfficientNetB0 and train at 100x smaller LR.
This adapts high-level features to our domain without destroying low-level ImageNet features.

In [ ]:
for layer in base_model.layers[-UNFREEZE_N:]:
    layer.trainable = True

trainable = sum(1 for l in model.layers if l.trainable)
print(f'Trainable layers after unfreeze: {trainable}')

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

print('Phase 2: fine-tuning top layers...')
history_p2 = model.fit(
    train_gen,
    validation_data=valid_gen,
    epochs=EPOCHS_P2,
    callbacks=get_callbacks('best_model.keras'),
    verbose=1,
)

## 11. Training Curves

In [ ]:
def plot_training_history(h1, h2, save_path=None):
    boundary = len(h1.history['loss'])
    metrics = {
        'Accuracy': ('accuracy', 'val_accuracy'),
        'Loss':     ('loss',     'val_loss'),
    }
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (title, (tk, vk)) in zip(axes, metrics.items()):
        train_v = h1.history[tk] + h2.history[tk]
        val_v   = h1.history[vk] + h2.history[vk]
        ax.plot(train_v, label='Train',      linewidth=2)
        ax.plot(val_v,   label='Validation', linewidth=2)
        ax.axvline(boundary, color='gray', linestyle='--',
                   linewidth=1.5, label='Fine-tune start')
        ax.set_title(f'{title} over Epochs', fontsize=12)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(title)
        ax.legend()
        ax.grid(alpha=0.3)
    plt.suptitle('Training History (Phase 1 + Phase 2)', fontsize=13)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

plot_training_history(history_p1, history_p2,
                      save_path=FIG_DIR / 'training_curves.png')

## 12. Model Evaluation

We go beyond accuracy because it can be misleading on imbalanced classes.
In aviation safety, **recall for crack** matters most: missing a crack is far worse than a false alarm.

In [ ]:
def evaluate_model(model, test_gen, classes, save_dir=None):
    test_gen.reset()
    y_prob = model.predict(test_gen, verbose=1).flatten()
    y_pred = (y_prob > 0.5).astype(int)
    y_true = test_gen.classes

    print('\n' + '='*55)
    print(' Classification Report')
    print('='*55)
    print(classification_report(y_true, y_pred, target_names=classes))

    cm = confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=axes[0],
                linewidths=0.5, linecolor='gray')
    axes[0].set_xlabel('Predicted', fontsize=11)
    axes[0].set_ylabel('Actual',    fontsize=11)
    axes[0].set_title('Confusion Matrix', fontsize=12)

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.3f}')
    axes[1].plot([0,1],[0,1], 'k--', lw=1)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve', fontsize=12)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    if save_dir:
        plt.savefig(save_dir / 'evaluation.png', dpi=150)
    plt.show()
    return y_true, y_pred, y_prob

y_true, y_pred, y_prob = evaluate_model(model, test_gen, CLASSES, save_dir=FIG_DIR)

## 13. Sample Predictions

In [ ]:
def visualize_predictions(model, test_gen, classes, n=12, save_path=None):
    test_gen.reset()
    images, labels = next(test_gen)
    n = min(n, len(images))
    preds = model.predict(images, verbose=0).flatten()

    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = axes.flat

    for i in range(n):
        pred_cls = int(preds[i] > 0.5)
        true_cls = int(labels[i])
        conf     = preds[i] if pred_cls == 1 else 1 - preds[i]
        color    = 'green' if pred_cls == true_cls else 'red'
        axes[i].imshow(images[i])
        axes[i].set_title(
            f'True: {classes[true_cls]}\nPred: {classes[pred_cls]} ({conf:.0%})',
            color=color, fontsize=9
        )
        axes[i].axis('off')
    for j in range(n, rows * cols):
        axes[j].axis('off')

    plt.suptitle('Sample Predictions  (green=correct | red=wrong)',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

visualize_predictions(model, test_gen, CLASSES,
                      save_path=FIG_DIR / 'sample_predictions.png')

## 14. Error Analysis

High-confidence wrong predictions are the most informative failures.
They reveal systematic blind spots in the model.

In [ ]:
def error_analysis(model, test_gen, classes, save_path=None):
    test_gen.reset()
    all_imgs, all_labels, all_probs = [], [], []
    for imgs, lbls in test_gen:
        probs = model.predict(imgs, verbose=0).flatten()
        all_imgs.append(imgs)
        all_labels.append(lbls)
        all_probs.append(probs)
        if sum(len(l) for l in all_labels) >= test_gen.n:
            break

    images = np.concatenate(all_imgs)[:test_gen.n]
    labels = np.concatenate(all_labels)[:test_gen.n]
    probs  = np.concatenate(all_probs)[:test_gen.n]
    preds  = (probs > 0.5).astype(int)

    wrong_mask = preds != labels.astype(int)
    confidence = np.where(preds == 1, probs, 1 - probs)
    n_wrong = wrong_mask.sum()

    print(f'Total test images : {test_gen.n}')
    print(f'Correct           : {test_gen.n - n_wrong}')
    print(f'Errors            : {n_wrong}  ({n_wrong/test_gen.n:.1%})')
    print(f'  FP (dent predicted as crack): {((preds==1)&(labels==0)).sum()}')
    print(f'  FN (crack predicted as dent): {((preds==0)&(labels==1)).sum()}')

    if n_wrong == 0:
        print('No errors found!')
        return

    wrong_idx = np.where(wrong_mask)[0][np.argsort(-confidence[wrong_mask])]
    show_n = min(8, n_wrong)
    cols = 4
    rows = (show_n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = axes.flat

    for i, idx in enumerate(wrong_idx[:show_n]):
        axes[i].imshow(images[idx])
        axes[i].set_title(
            f'True: {classes[int(labels[idx])]}\n'
            f'Pred: {classes[preds[idx]]} ({confidence[idx]:.0%} conf)',
            color='red', fontsize=9
        )
        axes[i].axis('off')
    for j in range(show_n, rows * cols):
        axes[j].axis('off')

    plt.suptitle('High-Confidence Errors', fontsize=12, color='darkred')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

error_analysis(model, test_gen, CLASSES,
               save_path=FIG_DIR / 'error_analysis.png')

## 15. Image Captioning with BLIP

We use **BLIP** (Bootstrapping Language-Image Pre-training) to generate natural-language descriptions of damage images.
This adds explainability: a maintenance engineer can read a plain-English description alongside the classification result.

In [ ]:
import torch
from PIL import Image as PILImage
from transformers import BlipProcessor, BlipForConditionalGeneration

def load_blip(model_name='Salesforce/blip-image-captioning-base'):
    print(f'Loading BLIP: {model_name}')
    processor  = BlipProcessor.from_pretrained(model_name)
    blip_model = BlipForConditionalGeneration.from_pretrained(model_name)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    blip_model = blip_model.to(device)
    blip_model.eval()
    print(f'BLIP loaded on {device}')
    return processor, blip_model, device

processor, blip_model, device = load_blip()

In [ ]:
def generate_caption(image_path, processor, model, device, prompt=None):
    img = PILImage.open(image_path).convert('RGB')
    inputs = processor(img, prompt, return_tensors='pt').to(device) if prompt else processor(img, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=50)
    return processor.decode(out[0], skip_special_tokens=True)

def caption_samples(test_dir, classes, n_per_class=2, save_path=None):
    samples = []
    for cls in classes:
        for img_path in sorted((test_dir / cls).glob('*.jpg'))[:n_per_class]:
            caption = generate_caption(img_path, processor, blip_model, device)
            samples.append((img_path, cls, caption))

    fig, axes = plt.subplots(len(samples), 2, figsize=(12, len(samples) * 3.5),
                             gridspec_kw={'width_ratios': [1, 2]})
    for i, (img_path, cls, caption) in enumerate(samples):
        axes[i][0].imshow(PILImage.open(img_path))
        axes[i][0].set_title(f'Class: {cls}', fontsize=10, fontweight='bold')
        axes[i][0].axis('off')
        axes[i][1].text(0.05, 0.5, f'"{caption}"',
                        transform=axes[i][1].transAxes,
                        fontsize=11, va='center', wrap=True,
                        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        axes[i][1].axis('off')

    plt.suptitle('BLIP-Generated Damage Captions', fontsize=13)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

caption_samples(TEST_DIR, CLASSES, save_path=FIG_DIR / 'blip_captions.png')

## 16. Save Final Model

In [ ]:
final_path = MODEL_DIR / 'aircraft_damage_final.keras'
model.save(str(final_path))
print(f'Model saved to: {final_path}')

## 17. Results Summary & Future Work

### What was built
- EfficientNetB0 two-phase transfer learning classifier (crack vs. dent)
- Rich augmentation pipeline tuned for small industrial datasets
- Full evaluation: accuracy, precision, recall, F1, confusion matrix, ROC-AUC
- Error analysis surfacing high-confidence failure cases
- BLIP natural-language captioning for explainability

### Future Work
- Expand to multi-class (corrosion, missing fastener, paint damage)
- Grad-CAM heatmaps for visual explainability
- REST API deployment (FastAPI + Docker) for MRO system integration
- Semi-supervised learning to leverage unlabeled inspection images
- Severity scoring (minor / moderate / critical) beyond binary classification